In [ ]:
# ============================================================
# AI-POWERED MULTILINGUAL TEXT TRANSLATION & ANALYSIS SYSTEM
# Google Colab - One Cell Version
# ============================================================

!pip -q install flask deep-translator textblob langdetect

import re
import threading
import time
import subprocess
from flask import Flask, request, jsonify, render_template_string
from deep_translator import GoogleTranslator
from langdetect import detect, LangDetectException
from textblob import TextBlob
from collections import Counter

# ------------------------------------------------------------
# Stop anything using port 5000
# ------------------------------------------------------------

subprocess.run(
    "fuser -k 5000/tcp",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# ------------------------------------------------------------
# Flask App
# ------------------------------------------------------------

app = Flask(__name__)

# ------------------------------------------------------------
# Language names
# ------------------------------------------------------------

LANGUAGES = {
    "English": "en",
    "Tamil": "ta",
    "Hindi": "hi",
    "Malayalam": "ml",
    "Telugu": "te",
    "Kannada": "kn",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Italian": "it",
    "Portuguese": "pt",
    "Japanese": "ja",
    "Korean": "ko",
    "Chinese": "zh-CN",
    "Arabic": "ar",
    "Russian": "ru",
    "Bengali": "bn",
    "Gujarati": "gu",
    "Marathi": "mr",
    "Punjabi": "pa",
    "Urdu": "ur"
}

# ------------------------------------------------------------
# Detect language
# ------------------------------------------------------------

def detect_language(text):

    try:
        code = detect(text)

        reverse = {v: k for k, v in LANGUAGES.items()}

        if code in reverse:
            return reverse[code], code

        return code.upper(), code

    except LangDetectException:
        return "Unknown", "auto"

    except Exception:
        return "Unknown", "auto"


# ------------------------------------------------------------
# Translation
# ------------------------------------------------------------

def translate_text(text, source, target):

    try:

        if source == target:
            return text

        translator = GoogleTranslator(
            source=source,
            target=target
        )

        return translator.translate(text)

    except Exception as e:

        return "Translation unavailable: " + str(e)


# ------------------------------------------------------------
# Sentiment
# ------------------------------------------------------------

def analyze_sentiment(text):

    try:

        blob = TextBlob(text)

        polarity = round(blob.sentiment.polarity, 3)
        subjectivity = round(blob.sentiment.subjectivity, 3)

        if polarity > 0.10:
            sentiment = "Positive"

        elif polarity < -0.10:
            sentiment = "Negative"

        else:
            sentiment = "Neutral"

        return sentiment, polarity, subjectivity

    except Exception:

        return "Unknown", 0, 0


# ------------------------------------------------------------
# Keyword extraction
# ------------------------------------------------------------

def extract_keywords(text):

    stopwords = {
        "the", "is", "a", "an", "and", "or", "to", "of",
        "in", "on", "for", "with", "this", "that", "it",
        "was", "are", "am", "i", "you", "we", "they",
        "he", "she", "my", "your", "our", "their",
        "be", "been", "have", "has", "had", "from",
        "as", "at", "by", "but", "very", "can", "will",
        "would", "should", "about", "into", "also"
    }

    words = re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())

    filtered = [
        word for word in words
        if word not in stopwords
    ]

    counts = Counter(filtered)

    return [word for word, count in counts.most_common(8)]


# ------------------------------------------------------------
# Text statistics
# ------------------------------------------------------------

def text_statistics(text):

    words = re.findall(r"\b\w+\b", text)

    sentences = [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

    return {
        "words": len(words),
        "characters": len(text),
        "sentences": len(sentences)
    }


# ------------------------------------------------------------
# Main analysis
# ------------------------------------------------------------

def analyze_text(text, target_language):

    detected_name, detected_code = detect_language(text)

    target_code = LANGUAGES.get(
        target_language,
        "en"
    )

    # Translate to English for analysis
    english_text = translate_text(
        text,
        detected_code,
        "en"
    )

    # Translate to selected target
    translated_text = translate_text(
        text,
        detected_code,
        target_code
    )

    # Sentiment
    sentiment, polarity, subjectivity = analyze_sentiment(
        english_text
    )

    # Keywords
    keywords = extract_keywords(english_text)

    # Statistics
    stats = text_statistics(text)

    return {
        "detected_language": detected_name,
        "detected_code": detected_code,
        "translated_text": translated_text,
        "english_text": english_text,
        "sentiment": sentiment,
        "polarity": polarity,
        "subjectivity": subjectivity,
        "keywords": keywords,
        "words": stats["words"],
        "characters": stats["characters"],
        "sentences": stats["sentences"]
    }


# ------------------------------------------------------------
# HTML
# ------------------------------------------------------------

HTML = """
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>AI Multilingual Text Intelligence</title>

<style>

* {
    box-sizing: border-box;
}

body {

    margin: 0;

    font-family:
        Arial,
        Helvetica,
        sans-serif;

    background:
        linear-gradient(
            135deg,
            #171c55,
            #54238c
        );

    min-height: 100vh;

    color: #222;
}

.container {

    width: 92%;

    max-width: 1050px;

    margin: auto;

    padding: 35px 0 60px;
}

.header {

    text-align: center;

    color: white;

    margin-bottom: 25px;
}

.header h1 {

    font-size: 36px;

    margin: 10px 0;

}

.header p {

    font-size: 17px;

    opacity: 0.9;

}

.card {

    background: white;

    border-radius: 20px;

    padding: 28px;

    margin-bottom: 22px;

    box-shadow:
        0 10px 30px rgba(0,0,0,0.18);

}

.card h2 {

    margin-top: 0;

    font-size: 22px;

}

textarea {

    width: 100%;

    height: 180px;

    padding: 16px;

    border: 2px solid #ddd;

    border-radius: 12px;

    resize: vertical;

    font-size: 16px;

    outline: none;

}

textarea:focus {

    border-color: #7040e8;

}

.controls {

    display: flex;

    flex-wrap: wrap;

    gap: 12px;

    margin-top: 15px;

    align-items: center;

}

select {

    padding: 13px;

    border-radius: 10px;

    border: 1px solid #ccc;

    font-size: 15px;

}

button {

    border: none;

    padding: 13px 20px;

    border-radius: 10px;

    color: white;

    font-size: 15px;

    font-weight: bold;

    cursor: pointer;

}

.speak {

    background: #16a34a;

}

.analyze {

    background: #7138e8;

}

.clear {

    background: #dc2626;

}

button:hover {

    opacity: 0.88;

}

.info {

    margin-top: 15px;

    padding: 13px;

    background: #eef5ff;

    border-radius: 10px;

    color: #1e40af;

}

.loading {

    display: none;

    text-align: center;

    padding: 20px;

    font-weight: bold;

}

.results {

    display: none;

}

.result-box {

    background: #f8f9ff;

    border-radius: 14px;

    padding: 18px;

    margin-bottom: 15px;

    border: 1px solid #e4e4ee;

}

.result-box h3 {

    margin-top: 0;

}

.translation {

    font-size: 18px;

    line-height: 1.6;

}

.stats {

    display: grid;

    grid-template-columns:
        repeat(auto-fit, minmax(150px, 1fr));

    gap: 12px;

}

.stat {

    background: #f1f3ff;

    padding: 18px;

    border-radius: 12px;

    text-align: center;

}

.stat strong {

    display: block;

    font-size: 26px;

    margin-top: 5px;

}

.sentiment {

    font-size: 22px;

    font-weight: bold;

}

.tags {

    display: flex;

    flex-wrap: wrap;

    gap: 8px;

}

.tag {

    background: #e9ddff;

    color: #5720a5;

    padding: 8px 12px;

    border-radius: 20px;

}

.error {

    background: #fee2e2;

    color: #991b1b;

    padding: 15px;

    border-radius: 10px;

}

.footer {

    text-align: center;

    color: white;

    margin-top: 25px;

    opacity: 0.8;

}

@media(max-width:600px) {

    .header h1 {
        font-size: 28px;
    }

    .card {
        padding: 20px;
    }

}

</style>

</head>


<body>

<div class="container">

    <div class="header">

        <h1>🌐 AI Multilingual Text Intelligence</h1>

        <p>
            Translation • Language Detection • Sentiment
            • Keywords • Speech Analysis
        </p>

    </div>


    <div class="card">

        <h2>📝 Enter Multilingual Text</h2>

        <textarea
            id="text"
            placeholder="Enter text in Tamil, Hindi, Malayalam, English, Telugu, Kannada, French, Spanish, etc..."
        ></textarea>


        <div class="controls">

            <button
                class="speak"
                onclick="startSpeech()"
            >
                🎤 Speak
            </button>


            <select id="targetLanguage">

                <option value="English">
                    🇬🇧 English
                </option>

                <option value="Tamil">
                    🇮🇳 Tamil
                </option>

                <option value="Hindi">
                    🇮🇳 Hindi
                </option>

                <option value="Malayalam">
                    🇮🇳 Malayalam
                </option>

                <option value="Telugu">
                    🇮🇳 Telugu
                </option>

                <option value="Kannada">
                    🇮🇳 Kannada
                </option>

                <option value="French">
                    🇫🇷 French
                </option>

                <option value="Spanish">
                    🇪🇸 Spanish
                </option>

                <option value="German">
                    🇩🇪 German
                </option>

                <option value="Japanese">
                    🇯🇵 Japanese
                </option>

                <option value="Korean">
                    🇰🇷 Korean
                </option>

                <option value="Arabic">
                    🇸🇦 Arabic
                </option>

            </select>


            <button
                class="analyze"
                onclick="analyzeText()"
            >
                🌍 Translate & Analyze
            </button>


            <button
                class="clear"
                onclick="clearText()"
            >
                🗑️ Clear
            </button>

        </div>


        <div class="info">

            💡 The system detects the input language,
            translates the text, and performs sentiment,
            keyword and statistical analysis.

        </div>

    </div>


    <div
        class="loading"
        id="loading"
    >
        ⏳ Processing your text...
    </div>


    <div
        class="card results"
        id="results"
    >

        <h2>📊 Analysis Results</h2>


        <div class="result-box">

            <h3>🌐 Detected Language</h3>

            <div id="language"></div>

        </div>


        <div class="result-box">

            <h3>🔄 Translation</h3>

            <div
                class="translation"
                id="translation"
            ></div>

        </div>


        <div class="result-box">

            <h3>🧠 English Text Used for Analysis</h3>

            <div id="englishText"></div>

        </div>


        <div class="result-box">

            <h3>😊 Sentiment</h3>

            <div
                class="sentiment"
                id="sentiment"
            ></div>

            <p id="sentimentDetails"></p>

        </div>


        <div class="result-box">

            <h3>🔑 Keywords</h3>

            <div
                class="tags"
                id="keywords"
            ></div>

        </div>


        <div class="result-box">

            <h3>📈 Text Statistics</h3>

            <div class="stats">

                <div class="stat">
                    Words
                    <strong id="words">0</strong>
                </div>

                <div class="stat">
                    Characters
                    <strong id="characters">0</strong>
                </div>

                <div class="stat">
                    Sentences
                    <strong id="sentences">0</strong>
                </div>

            </div>

        </div>

    </div>


    <div class="footer">

        AI-Powered Multilingual Text Translation &
        Analysis System

    </div>

</div>


<script>

function startSpeech() {

    const SpeechRecognition =
        window.SpeechRecognition ||
        window.webkitSpeechRecognition;

    if (!SpeechRecognition) {

        alert(
            "Speech recognition is not supported. Please use Google Chrome."
        );

        return;
    }

    const recognition =
        new SpeechRecognition();

    recognition.lang = "en-IN";

    recognition.continuous = false;

    recognition.interimResults = false;


    recognition.onstart = function() {

        alert(
            "🎤 Speak now..."
        );

    };


    recognition.onresult = function(event) {

        const result =
            event.results[0][0].transcript;

        document.getElementById("text").value =
            result;

    };


    recognition.onerror = function(event) {

        alert(
            "Speech recognition error: " +
            event.error
        );

    };


    recognition.start();

}


async function analyzeText() {

    const text =
        document.getElementById("text").value.trim();

    const targetLanguage =
        document.getElementById("targetLanguage").value;


    if (!text) {

        alert(
            "Please enter some text first."
        );

        return;
    }


    document.getElementById("loading").style.display =
        "block";

    document.getElementById("results").style.display =
        "none";


    try {

        const response =
            await fetch(
                "/analyze",
                {

                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({
                        text: text,
                        target_language:
                            targetLanguage
                    })

                }
            );


        const data =
            await response.json();


        if (!response.ok) {

            throw new Error(
                data.error ||
                "Something went wrong."
            );

        }


        document.getElementById("language").innerHTML =
            "🌐 <strong>" +
            data.detected_language +
            "</strong> (" +
            data.detected_code +
            ")";


        document.getElementById("translation").innerText =
            data.translated_text;


        document.getElementById("englishText").innerText =
            data.english_text;


        document.getElementById("sentiment").innerText =
            data.sentiment;


        document.getElementById("sentimentDetails").innerText =
            "Polarity: " +
            data.polarity +
            " | Subjectivity: " +
            data.subjectivity;


        const keywordBox =
            document.getElementById("keywords");

        keywordBox.innerHTML = "";


        data.keywords.forEach(
            function(keyword) {

                const tag =
                    document.createElement("span");

                tag.className =
                    "tag";

                tag.innerText =
                    keyword;

                keywordBox.appendChild(tag);

            }
        );


        document.getElementById("words").innerText =
            data.words;

        document.getElementById("characters").innerText =
            data.characters;

        document.getElementById("sentences").innerText =
            data.sentences;


        document.getElementById("results").style.display =
            "block";


        document.getElementById("results").scrollIntoView({
            behavior: "smooth"
        });

    }

    catch(error) {

        alert(
            "Error: " +
            error.message
        );

    }

    finally {

        document.getElementById("loading").style.display =
            "none";

    }

}


function clearText() {

    document.getElementById("text").value = "";

    document.getElementById("results").style.display =
        "none";

}

</script>

</body>

</html>
"""


# ------------------------------------------------------------
# Routes
# ------------------------------------------------------------

@app.route("/")
def home():

    return render_template_string(HTML)


@app.route("/analyze", methods=["POST"])
def analyze():

    try:

        data = request.get_json()

        text = data.get("text", "").strip()

        target_language = data.get(
            "target_language",
            "English"
        )

        if not text:

            return jsonify({
                "error": "Text is empty."
            }), 400

        result = analyze_text(
            text,
            target_language
        )

        return jsonify(result)

    except Exception as e:

        return jsonify({
            "error": str(e)
        }), 500


# ------------------------------------------------------------
# Start Flask
# ------------------------------------------------------------

def run_app():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


thread = threading.Thread(
    target=run_app,
    daemon=True
)

thread.start()

time.sleep(3)


# ------------------------------------------------------------
# Open Colab browser URL
# ------------------------------------------------------------

from google.colab import output

url = output.eval_js(
    "google.colab.kernel.proxyPort(5000)"
)

print("=" * 60)
print("AI MULTILINGUAL TEXT INTELLIGENCE SYSTEM")
print("=" * 60)
print("Server started successfully!")
print()
print("OPEN THIS LINK:")
print(url)
print("=" * 60)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


AI MULTILINGUAL TEXT INTELLIGENCE SYSTEM
Server started successfully!

OPEN THIS LINK:
https://5000-m-s-kkb-use1b2-2x6wvrbckpbjh-b.us-east1-2.prod.colab.dev
